# `get_mycoportal_synonyms` tests

`get_mycoportal_synonyms(species_name)` returns a dict whose keys are species-level synonym names and values are empty lists.

It uses three steps internally:
1. **gettaxasuggest RPC** — convert the species name to a taxon ID (tid)
2. **REST API** — resolve to the accepted taxon's tid if the input is itself a synonym
3. **HTML page** — scrape species-level synonyms from the portal's `synonymDiv`

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

from scripts.APIs.MyCoPortal import get_mycoportal_synonyms

## Species with multiple synonyms

In [2]:
get_mycoportal_synonyms("Gymnopus dryophilus")

{'Gymnopus dryophilus': [],
 'Agaricus dryophilus': [],
 'Collybia dryophila': [],
 'Marasmius dryophilus': [],
 'Omphalia dryophila': []}

The website for this query: [https://mycoportal.org/portal/taxa/index.php?taxon=Gymnopus+dryophilus#](https://mycoportal.org/portal/taxa/index.php?taxon=Gymnopus+dryophilus#)

Here's what's shown on that website, and I'll highlight what got selected by our script:

| Name (from MyCoPortal) | Included? | Reason |
|---|---|---|
| *Agaricus dryophilus* Bull. | ✅ Yes | Species-level synonym |
| *Collybia aquosa* var. *dryophila* (Bull.) Krieglst. | ❌ No | Infraspecific (var.) — only this variety was treated as equivalent, not the whole species *Collybia aquosa* |
| *Collybia dryophila* (Bull.) P. Kumm. | ✅ Yes | Species-level synonym |
| *Collybia dryophila* var. *alvearis* Cooke | ❌ No | Infraspecific (var.) |
| *Collybia dryophila* var. *aurata* Quél. | ❌ No | Infraspecific (var.) |
| *Collybia dryophila* var. *oedipoides* Singer | ❌ No | Infraspecific (var.) |
| *Collybia dryophila* var. *steinmannii* Raithelh. | ❌ No | Infraspecific (var.) |
| *Marasmius dryophilus* (Bull.) P. Karst. | ✅ Yes | Species-level synonym |
| *Marasmius dryophilus* var. *alvearis* (Cooke) Rea | ❌ No | Infraspecific (var.) |
| *Marasmius dryophilus* var. *auratus* (Quél.) Rea | ❌ No | Infraspecific (var.) |
| *Omphalia dryophila* (Bull.) Gray | ✅ Yes | Species-level synonym |


## Species with no species-level synonyms

MyCoPortal lists only infraspecific names (var., subsp., etc.) under *Amanita muscaria* — those are filtered out, leaving just the queried name.

In [3]:
get_mycoportal_synonyms("Amanita muscaria")

{'Amanita muscaria': []}

The website for this query: [https://mycoportal.org/portal/taxa/index.php?taxon=Amanita+muscaria#](https://mycoportal.org/portal/taxa/index.php?taxon=Amanita+muscaria#)

Here's what's shown on that website, and I'll highlight what got selected by our script:

| Name (from MyCoPortal) | Included? | Reason |
|---|---|---|
| *Amanita muscaria* var. *aureola* (Kalchbr.) Quél. | ❌ No | Infraspecific (var.) |
| *Amanita muscaria* var. *muscaria* (L.) Lam. | ❌ No | Infraspecific (var.) |


## Input is itself a synonym

*Agaricus dryophilus* is a synonym of *Gymnopus dryophilus*. The function resolves to the accepted taxon first, then returns all synonyms. Same synonym list as Gymnopus dryophilus — because querying a synonym resolves to the accepted taxon first, then scrapes the same page. The one difference is that Agaricus dryophilus itself appears in the scraped list but is deduplicated since it was already added as the queried name.

In [4]:
get_mycoportal_synonyms("Agaricus dryophilus")

{'Agaricus dryophilus': [],
 'Gymnopus dryophilus': [],
 'Collybia dryophila': [],
 'Marasmius dryophilus': [],
 'Omphalia dryophila': []}

The website for this query: [https://mycoportal.org/portal/taxa/index.php?taxon=Agaricus+dryophilus#](https://mycoportal.org/portal/taxa/index.php?taxon=Agaricus+dryophilus#)

Here's what's shown on that website, and I'll highlight what got selected by our script:

| Name (from MyCoPortal) | Included? | Reason |
|---|---|---|
| *Agaricus dryophilus* Bull. | ✅ Yes (deduplicated) | The queried name — already added before scraping; duplicate from the page is skipped |
| *Collybia aquosa* var. *dryophila* (Bull.) Krieglst. | ❌ No | Infraspecific (var.) — only this variety was treated as equivalent, not the whole species *Collybia aquosa* |
| *Collybia dryophila* (Bull.) P. Kumm. | ✅ Yes | Species-level synonym |
| *Collybia dryophila* var. *alvearis* Cooke | ❌ No | Infraspecific (var.) |
| *Collybia dryophila* var. *aurata* Quél. | ❌ No | Infraspecific (var.) |
| *Collybia dryophila* var. *oedipoides* Singer | ❌ No | Infraspecific (var.) |
| *Collybia dryophila* var. *steinmannii* Raithelh. | ❌ No | Infraspecific (var.) |
| *Marasmius dryophilus* (Bull.) P. Karst. | ✅ Yes | Species-level synonym |
| *Marasmius dryophilus* var. *alvearis* (Cooke) Rea | ❌ No | Infraspecific (var.) |
| *Marasmius dryophilus* var. *auratus* (Quél.) Rea | ❌ No | Infraspecific (var.) |
| *Omphalia dryophila* (Bull.) Gray | ✅ Yes | Species-level synonym |



## Species not found

In [5]:
get_mycoportal_synonyms("Totally fakeus speciesname")

{}

The website for this query: [https://mycoportal.org/portal/taxa/index.php?taxon=Totally+fakeus+speciesname#](https://mycoportal.org/portal/taxa/index.php?taxon=Totally+fakeus+speciesname#)

The webpage returns "Totally fakeus speciesname not found"